# 10.1 Documentación HTML en la web → grafo (Ungraph)

Este notebook ejemplifica el flujo probado por el script `scripts/verify_documentation_ingest_pipeline.py` y el módulo `ungraph.utils.verify_doc_pipeline`:

1. **HTTP** → HTML crudo  
2. **CIR** (`extract_web_document`) → bloques con provenance  
3. **Markdown outline** → `LangChainDocumentLoaderService`  
4. **Chunking** `markdown_header` → chunks con `page_number` lógico  
5. **Neo4j** con patrón `FILE_PAGE_CHUNK` vía `ungraph.ingest_document(..., source_url=...)` (sección 4; credenciales desde `.env`)

## Cuatro semillas de documentación

| # | Nombre | URL |
|---|--------|-----|
| 1 | IBM Quantum (guides) | https://quantum.cloud.ibm.com/docs/en/guides |
| 2 | LangChain Reference (home) | https://reference.langchain.com/ |
| 3 | Neo4j Documentation (hub) | https://neo4j.com/docs/ |
| 4 | LangChain Reference (Python) | https://reference.langchain.com/python |

**Requisitos:** `pip install 'ungraph[crawl]'` (httpx). **Persistencia:** Neo4j accesible y un archivo `.env` en el proyecto (o cwd) con `UNGRAPH_NEO4J_URI`, `UNGRAPH_NEO4J_PASSWORD`, etc. El primer bloque de código carga variables con `python-dotenv` (dependencia de Ungraph).


In [1]:
# pip install -e ".[crawl]"  (httpx)
from dotenv import find_dotenv, load_dotenv

# Carga UNGRAPH_* (Neo4j, etc.) antes de cualquier import que fije la configuración
load_dotenv(find_dotenv())

from ungraph.utils.verify_doc_pipeline import (
    DOCUMENTATION_SEEDS,
    DocSeed,
    step_httpx_and_cir,
    step_loader_and_chunking,
    run_verification,
)

for s in DOCUMENTATION_SEEDS:
    print(f"• {s.name}: {s.url}")


d:\projects\Ungraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


• IBM Quantum (guides): https://quantum.cloud.ibm.com/docs/en/guides
• LangChain Reference (home): https://reference.langchain.com/
• Neo4j Documentation (hub): https://neo4j.com/docs/
• LangChain Reference (Python): https://reference.langchain.com/python


## 1. Una página: CIR + chunking (sin Neo4j)

Replica lo que hace el script de verificación para una URL.

In [2]:
import httpx

url = DOCUMENTATION_SEEDS[0].url
r = httpx.get(url, timeout=45.0, follow_redirects=True, headers={"User-Agent": "UngraphNotebook/1.0"})
r.raise_for_status()

n_blocks, preview = step_httpx_and_cir(url)
n_chunks = step_loader_and_chunking(r.content, "page.html", source_url=str(r.url))
print("Bloques CIR:", n_blocks, "| Chunks:", n_chunks)
print("Preview MD:\n", preview[:500])


2026-04-12 23:07:44,892 - INFO - Cargando HTML: C:\Users\jefai\AppData\Local\Temp\tmpuq9xy7e1\page.html
2026-04-12 23:07:46,081 - INFO - HTML convertido a markdown outline para ingestión
2026-04-12 23:07:46,082 - INFO - Smart chunking document: page.html (preferred=markdown_header)
2026-04-12 23:07:46,083 - INFO - Tipo de documento detectado: html
2026-04-12 23:07:46,085 - INFO - Estructura: 5 headers, 48 párrafos, 450 palabras
2026-04-12 23:07:46,085 - INFO - Usando estrategia preferida: markdown_header


Bloques CIR: 48 | Chunks: 8
Preview MD:
 # Introduction

## Introduction to Qiskit and IBM Quantum

Welcome to the documentation for Qiskit, its related packages, and IBM Quantum® Platform. This documentation includes how-to guides to get you started on our tools, specific use-case tutorials that include end-to-end examples, and a collection of API references.

Qiskit provides a modular and extensible framework for quantum research and d


## 2. Verificación completa (las 4 semillas)

Equivale a: `uv run python scripts/verify_documentation_ingest_pipeline.py`

In [3]:
summary = run_verification(ingest_one=False)
for row in summary["seeds"]:
    print(row["name"], "→ CIR:", row["cir_blocks"], "chunks:", row["chunks"])


2026-04-12 23:07:46,099 - INFO - === IBM Quantum (guides) ===
2026-04-12 23:07:46,595 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/en/guides "HTTP/1.1 200 OK"
2026-04-12 23:07:46,891 - INFO -   CIR bloques: 48
2026-04-12 23:07:47,791 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/en/guides "HTTP/1.1 200 OK"
2026-04-12 23:07:47,928 - INFO - Cargando HTML: C:\Users\jefai\AppData\Local\Temp\tmpmwz0yq2j\page.html
2026-04-12 23:07:48,571 - INFO - HTML convertido a markdown outline para ingestión
2026-04-12 23:07:48,572 - INFO - Smart chunking document: page.html (preferred=markdown_header)
2026-04-12 23:07:48,573 - INFO - Tipo de documento detectado: html
2026-04-12 23:07:48,574 - INFO - Estructura: 5 headers, 48 párrafos, 450 palabras
2026-04-12 23:07:48,574 - INFO - Usando estrategia preferida: markdown_header
2026-04-12 23:07:48,577 - INFO -   Chunks (markdown_header): 8
2026-04-12 23:07:48,578 - INFO - === LangChain Reference (home) ===
2026-04-12 23:07:4

IBM Quantum (guides) → CIR: 48 chunks: 8
LangChain Reference (home) → CIR: 12 chunks: 5
Neo4j Documentation (hub) → CIR: 80 chunks: 16
LangChain Reference (Python) → CIR: 12 chunks: 5


## 3. Crawl multi-página (BFS + prefijo)

Usa `DocumentationCrawlConfig` y `crawl_documentation_site` para descargar varias páginas a disco. Luego cada HTML se puede ingerir con `source_url`.

CLI equivalente: `python scripts/crawl_docs_to_graph.py --preset ibm-quantum --max-pages 5 --out ./_crawl --dry-run`

In [4]:
from pathlib import Path
from ungraph.infrastructure.services.doc_site_crawler import (
    DocumentationCrawlConfig,
    crawl_documentation_site,
)

out = Path("_notebook_crawl_demo")
out.mkdir(exist_ok=True)
cfg = DocumentationCrawlConfig(
    seed_urls=["https://quantum.cloud.ibm.com/docs/en/guides"],
    max_pages=3,
    path_prefix="/docs/en",
    delay_seconds=0.5,
    use_sitemap=True,
    sitemap_url="https://quantum.cloud.ibm.com/sitemap.xml",
)
pages = crawl_documentation_site(cfg, out)
print(f"Descargadas {len(pages)} páginas en {out}")
for p in pages[:5]:
    print(" ", p.url)


2026-04-12 23:07:52,040 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/sitemap.xml "HTTP/1.1 200 OK"
2026-04-12 23:07:53,092 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/sitemap-platform.xml "HTTP/1.1 200 OK"
2026-04-12 23:07:53,849 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/sitemap-0.xml "HTTP/1.1 200 OK"
2026-04-12 23:07:54,491 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/learning/sitemap-0.xml "HTTP/1.1 200 OK"
2026-04-12 23:07:55,135 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/en/api "HTTP/1.1 200 OK"
2026-04-12 23:07:56,004 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/en/errors "HTTP/1.1 200 OK"
2026-04-12 23:07:57,021 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/en/api/qiskit-runtime-rest/changelog "HTTP/1.1 200 OK"


Descargadas 3 páginas en _notebook_crawl_demo
  https://quantum.cloud.ibm.com/docs/en/api
  https://quantum.cloud.ibm.com/docs/en/errors
  https://quantum.cloud.ibm.com/docs/en/api/qiskit-runtime-rest/changelog


## 4. Persistir en Neo4j

Define en `.env` (o en el entorno) al menos `UNGRAPH_NEO4J_URI` y `UNGRAPH_NEO4J_PASSWORD`. La celda siguiente vuelve a cargar `.env`, reinicia la configuración de Ungraph y ejecuta `run_verification(ingest_one=True)` (equivale a `verify_documentation_ingest_pipeline.py --ingest-one`).

Alternativa manual:

```python
import ungraph
ungraph.configure(neo4j_uri="bolt://localhost:7687", neo4j_password="...")
ungraph.ingest_document(path_a_html, source_url="https://...")
```

El patrón por defecto es **FILE_PAGE_CHUNK** (File → Page → Chunk; `Page` = sección lógica para HTML).

In [5]:
from dotenv import find_dotenv, load_dotenv

from ungraph.core.configuration import reset_configuration

# Por si el kernel ya importó ungraph antes: recargar .env y refrescar Settings
load_dotenv(find_dotenv())
reset_configuration()

from ungraph.utils.verify_doc_pipeline import run_verification

run_verification(ingest_one=True)


2026-04-12 23:07:57,689 - INFO - === IBM Quantum (guides) ===
2026-04-12 23:07:58,328 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/en/guides "HTTP/1.1 200 OK"
2026-04-12 23:07:58,484 - INFO -   CIR bloques: 48
2026-04-12 23:07:58,974 - INFO - HTTP Request: GET https://quantum.cloud.ibm.com/docs/en/guides "HTTP/1.1 200 OK"
2026-04-12 23:07:59,274 - INFO - Cargando HTML: C:\Users\jefai\AppData\Local\Temp\tmpvc0j2e5_\page.html
2026-04-12 23:07:59,938 - INFO - HTML convertido a markdown outline para ingestión
2026-04-12 23:07:59,939 - INFO - Smart chunking document: page.html (preferred=markdown_header)
2026-04-12 23:07:59,940 - INFO - Tipo de documento detectado: html
2026-04-12 23:07:59,940 - INFO - Estructura: 5 headers, 48 párrafos, 450 palabras
2026-04-12 23:07:59,941 - INFO - Usando estrategia preferida: markdown_header
2026-04-12 23:07:59,946 - INFO -   Chunks (markdown_header): 8
2026-04-12 23:07:59,947 - INFO - === LangChain Reference (home) ===
2026-04-12 23:08:0

RuntimeError: Neo4j no configurado: UNGRAPH_NEO4J_URI y UNGRAPH_NEO4J_PASSWORD o ungraph.configure(...)